# Examen Machine Learning — Master IA (M1)
## Thématique : Marketing/E-commerce — Analyse du Désabonnement Client (Churn)
### Dataset : Telco Customer Churn
**FST 2025-2026**

---
Ce notebook couvre les 4 phases de l'examen :
1. Exploration et Prétraitement
2. Apprentissage Supervisé (Régression + Classification + KNN)
3. Apprentissage Non Supervisé (K-Means + Fuzzy C-Means)
4. Analyse Comparative et Synthèse


In [ ]:
# ── Imports et Configuration ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_squared_error, r2_score, confusion_matrix,
                              classification_report, f1_score, accuracy_score,
                              adjusted_rand_score)
import skfuzzy as fuzz
import warnings
warnings.filterwarnings('ignore')

# Style global
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13,
                     'axes.titleweight': 'bold', 'figure.dpi': 120})
COLORS = ['#2C3E8C', '#E63946', '#2A9D8F', '#F4A261', '#8338EC', '#06D6A0']
print("Bibliothèques chargées avec succès.")

---
## Phase 1 : Exploration et Prétraitement (20%)
### 1.1 Chargement et Analyse Descriptive

In [ ]:
# Chargement du dataset Telco Customer Churn
# Source : https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
import urllib.request
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
urllib.request.urlretrieve(url, 'telco_churn.csv')

df = pd.read_csv('telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"Dimensions du dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"\nTypes de variables :")
print(df.dtypes)
print(f"\nAperçu des 5 premières lignes :")
df.head()

In [ ]:
# Statistiques descriptives des variables numériques
df.describe()

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
print("Valeurs manquantes par colonne :")
print(missing[missing > 0])
print(f"\nNombre total de valeurs manquantes : {missing.sum()}")

# Gestion : imputation par la médiane pour TotalCharges
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df.dropna(inplace=True)
print(f"\nAprès imputation — Dataset propre : {df.shape[0]} lignes")

In [ ]:
# Distribution des variables numériques
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, c in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges'], COLORS):
    ax.hist(df[col], bins=30, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribution : {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Fréquence')
plt.suptitle('Distributions des Variables Numériques', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Interprétation :
# - tenure : bimodale (clients récents et clients fidèles)
# - MonthlyCharges : distribution quasi-uniforme avec pic sur les bas tarifs
# - TotalCharges : distribution asymétrique positive (majorité < 2000$)

In [ ]:
# Détection des valeurs aberrantes (Boxplots)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, c in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges'], COLORS):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor=c, alpha=0.7),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(f'Boxplot : {col}')
    ax.set_ylabel(col)
plt.suptitle('Détection des Valeurs Aberrantes', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Observation : pas d'outliers extrêmes — les données sont naturellement bornées

In [ ]:
# Distribution de la variable cible : Churn
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['Churn'].value_counts()
bars = ax.bar(counts.index, counts.values, color=[COLORS[0], COLORS[1]],
              edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Distribution de la Variable Cible : Churn')
ax.set_xlabel('Churn')
ax.set_ylabel('Nombre de clients')
plt.tight_layout()
plt.show()

# Observation : déséquilibre de classes — 73.5% No Churn vs 26.5% Churn
# Ce déséquilibre influencera l'évaluation des modèles (privilégier F1 plutôt qu'accuracy)

### 1.2 Analyse de Corrélation

In [ ]:
# Encodage des variables catégorielles
df_enc = df.copy()
df_enc.drop('customerID', axis=1, inplace=True)

le = LabelEncoder()
cat_cols = df_enc.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

print("Variables catégorielles encodées :", cat_cols)

# Normalisation des variables numériques
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df_enc[num_cols] = scaler.fit_transform(df_enc[num_cols])
print("Variables numériques normalisées (StandardScaler) :", num_cols)

In [ ]:
# Matrice de corrélation
fig, ax = plt.subplots(figsize=(14, 11))
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', ax=ax,
            cmap='coolwarm', center=0, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Matrice de Corrélation — Telco Customer Churn', pad=15)
plt.tight_layout()
plt.show()

# Variables les plus corrélées avec Churn :
top_corr = corr['Churn'].abs().sort_values(ascending=False).head(8)
print("\nTop 7 variables corrélées avec Churn :")
print(top_corr)

---
## Phase 2 : Apprentissage Supervisé (40%)
### 2.1 Régression Linéaire — Prédiction de MonthlyCharges

In [ ]:
# Problème de régression : prédire MonthlyCharges à partir des autres features
X_reg = df_enc.drop(['MonthlyCharges', 'Churn'], axis=1)
y_reg = df_enc['MonthlyCharges']

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
print(f"Train : {X_tr.shape[0]} échantillons | Test : {X_te.shape[0]} échantillons")

# Entraînement
lr = LinearRegression()
lr.fit(X_tr, y_tr)
y_pred_lr = lr.predict(X_te)

# Métriques
mse = mean_squared_error(y_te, y_pred_lr)
r2  = r2_score(y_te, y_pred_lr)
print(f"\n=== Résultats Régression Linéaire ===")
print(f"  MSE (Erreur Quadratique Moyenne) : {mse:.4f}")
print(f"  R²  (Coefficient de détermination) : {r2:.4f}")
print(f"  Interprétation : le modèle explique {r2*100:.1f}% de la variance des charges mensuelles")

In [ ]:
# Visualisation : Réel vs Prédit + Résidus
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter réel vs prédit
axes[0].scatter(y_te, y_pred_lr, alpha=0.4, color=COLORS[0], s=20)
mn, mx = float(y_te.min()), float(y_te.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Prédiction parfaite')
axes[0].set_title(f'Régression Linéaire — Réel vs Prédit\nMSE={mse:.4f}   R²={r2:.4f}')
axes[0].set_xlabel('Valeurs Réelles (normalisées)')
axes[0].set_ylabel('Valeurs Prédites (normalisées)')
axes[0].legend()

# Distribution des résidus
residuals = y_te - y_pred_lr
axes[1].hist(residuals, bins=40, color=COLORS[2], edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', lw=2, ls='--', label='Résidu nul')
axes[1].set_title('Distribution des Résidus')
axes[1].set_xlabel('Résidu')
axes[1].set_ylabel('Fréquence')
axes[1].legend()
plt.tight_layout()
plt.show()

### 2.2 Classification — Régression Logistique

In [ ]:
# Problème de classification binaire : prédire Churn (0/1)
X_clf = df_enc.drop('Churn', axis=1)
y_clf = df_enc['Churn']

# Stratified split pour respecter la distribution des classes
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print(f"Train : {X_tr2.shape[0]} | Test : {X_te2.shape[0]}")
print(f"Distribution Churn dans train : {y_tr2.value_counts(normalize=True).to_dict()}")

# Entraînement Régression Logistique
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_tr2, y_tr2)
y_pred_log = log_reg.predict(X_te2)

# Métriques
f1_log  = f1_score(y_te2, y_pred_log)
acc_log = accuracy_score(y_te2, y_pred_log)
print(f"\n=== Résultats Régression Logistique ===")
print(f"  Accuracy  : {acc_log:.4f}")
print(f"  F1-Score  : {f1_log:.4f}")
print(f"\n--- Rapport de Classification ---")
print(classification_report(y_te2, y_pred_log, target_names=['No Churn', 'Churn']))

In [ ]:
# Matrice de Confusion — Régression Logistique
cm_log = confusion_matrix(y_te2, y_pred_log)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'],
            linewidths=1, linecolor='white')
ax.set_title(f'Matrice de Confusion — Régression Logistique\nF1={f1_log:.4f}   Accuracy={acc_log:.4f}')
ax.set_ylabel('Réel')
ax.set_xlabel('Prédit')
plt.tight_layout()
plt.show()

# Analyse :
# TP (vrais positifs pour Churn), FP (faux positifs), FN (faux négatifs)
tn, fp, fn, tp = cm_log.ravel()
print(f"\nTP={tp} | FP={fp} | FN={fn} | TN={tn}")
print(f"Le modèle manque {fn} clients qui vont réellement churner (faux négatifs critiques)")

### 2.3 K-Nearest Neighbors (KNN) avec GridSearch

In [ ]:
# GridSearch pour trouver le k optimal
param_grid = {'n_neighbors': list(range(1, 21))}
knn = KNeighborsClassifier()
grid = GridSearchCV(knn, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_tr2, y_tr2)

best_k = grid.best_params_['n_neighbors']
print(f"Meilleur k trouvé par GridSearch (5-fold CV) : k = {best_k}")
print(f"F1-Score cross-validation : {grid.best_score_:.4f}")

# Entraînement du meilleur modèle
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_tr2, y_tr2)
y_pred_knn = knn_best.predict(X_te2)

f1_knn  = f1_score(y_te2, y_pred_knn)
acc_knn = accuracy_score(y_te2, y_pred_knn)
print(f"\n=== Résultats KNN (k={best_k}) ===")
print(f"  Accuracy  : {acc_knn:.4f}")
print(f"  F1-Score  : {f1_knn:.4f}")

In [ ]:
# Visualisation : Courbe GridSearch + Matrice de Confusion KNN
k_vals = list(range(1, 21))
cv_scores = list(grid.cv_results_['mean_test_score'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe F1 vs k
axes[0].plot(k_vals, cv_scores, 'o-', color=COLORS[0], lw=2, ms=6)
axes[0].axvline(best_k, color=COLORS[1], ls='--', lw=2, label=f'k optimal = {best_k}')
axes[0].fill_between(k_vals, cv_scores, alpha=0.1, color=COLORS[0])
axes[0].set_title('GridSearch KNN — Score F1 par k (CV=5)')
axes[0].set_xlabel('Nombre de voisins (k)')
axes[0].set_ylabel('F1-Score (Cross-Validation)')
axes[0].legend()

# Confusion Matrix
cm_knn = confusion_matrix(y_te2, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'],
            linewidths=1, linecolor='white')
axes[1].set_title(f'Matrice de Confusion — KNN (k={best_k})\nF1={f1_knn:.4f}   Accuracy={acc_knn:.4f}')
axes[1].set_ylabel('Réel')
axes[1].set_xlabel('Prédit')
plt.tight_layout()
plt.show()

---
## Phase 3 : Apprentissage Non Supervisé (30%)
### 3.1 K-Means — Méthode du Coude

In [ ]:
# Données sans étiquettes pour le clustering
X_clust = df_enc.drop('Churn', axis=1)
print(f"Données pour clustering : {X_clust.shape}")

# Méthode du coude
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)

# Visualisation
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_range), inertias, 'o-', color=COLORS[0], lw=2.5, ms=8)
ax.axvline(3, color=COLORS[1], ls='--', lw=2, label='k optimal = 3 (coude)')
ax.fill_between(list(K_range), inertias, alpha=0.1, color=COLORS[0])
ax.set_title('Méthode du Coude (Elbow Method) — Détermination du k optimal')
ax.set_xlabel('Nombre de clusters k')
ax.set_ylabel('Inertie Intra-cluster (WCSS)')
ax.legend()
plt.tight_layout()
plt.show()

print("Observation : le coude se forme à k=3, au-delà le gain en inertie est marginal.")

In [ ]:
# K-Means avec k=3 + Visualisation PCA
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = km3.fit_predict(X_clust)

# Réduction PCA 2D pour visualisation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_clust)
var_exp = pca.explained_variance_ratio_.sum() * 100

fig, ax = plt.subplots(figsize=(9, 6))
for i, c in enumerate(COLORS[:3]):
    mask = labels_km == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=c, alpha=0.5, s=15, label=f'Cluster {i+1}')
ax.set_title(f'K-Means (k=3) — Projection PCA 2D\nVariance expliquée : {var_exp:.1f}%')
ax.set_xlabel('Composante Principale 1')
ax.set_ylabel('Composante Principale 2')
ax.legend()
plt.tight_layout()
plt.show()

# Profil des clusters
df_profile = df.copy()
df_profile['Cluster'] = labels_km
print("\nProfil moyen des clusters K-Means :")
print(df_profile.groupby('Cluster')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean().round(1))

In [ ]:
# Visualisation du profil des clusters
df_profile = df.copy()
df_profile['Cluster'] = labels_km
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    vals = [df_profile[df_profile['Cluster'] == i][col].mean() for i in range(3)]
    bars = ax.bar([f'Cluster {i+1}' for i in range(3)], vals,
                  color=COLORS[:3], edgecolor='white')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
                f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'Moyenne {col} par Cluster')
    ax.set_ylabel(col)
plt.suptitle('Profil des Segments Clients — K-Means', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 Fuzzy C-Means (FCM)

In [ ]:
# Fuzzy C-Means avec skfuzzy
X_t = X_clust.values.T  # shape (features, samples) requis par skfuzzy

# Paramètres :
# c=3 : nombre de clusters (cohérent avec K-Means)
# m=2 : exposant de flou (fuzziness coefficient)
# error=0.005 : critère de convergence
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    X_t, c=3, m=2, error=0.005, maxiter=1000, init=None, seed=42)

labels_fcm = np.argmax(u, axis=0)  # cluster avec le degré d'appartenance le plus élevé
print(f"Fuzzy Partition Coefficient (FPC) : {fpc:.4f}")
print(f"  (FPC = 1 : clusters parfaitement séparés, FPC = 1/c : chevauchement total)")
print(f"\nDistribution des clusters FCM :")
unique, counts = np.unique(labels_fcm, return_counts=True)
for cl, cnt in zip(unique, counts):
    print(f"  Cluster {cl+1} : {cnt} clients ({cnt/len(labels_fcm)*100:.1f}%)")

In [ ]:
# Visualisation FCM : projection PCA + heatmap des appartenances
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter PCA
for i, c in enumerate(COLORS[:3]):
    mask = labels_fcm == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=c, alpha=0.5, s=15, label=f'Cluster {i+1}')
axes[0].set_title('Fuzzy C-Means (c=3) — Projection PCA 2D')
axes[0].set_xlabel('CP1'); axes[0].set_ylabel('CP2'); axes[0].legend()

# Heatmap des degrés d'appartenance (200 clients aléatoires)
np.random.seed(42)
idx = np.random.choice(X_clust.shape[0], 200, replace=False)
u_sample = u[:, idx]
sns.heatmap(u_sample, ax=axes[1], cmap='YlOrRd',
            yticklabels=['Cluster 1', 'Cluster 2', 'Cluster 3'],
            xticklabels=False,
            cbar_kws={'label': "Degré d'appartenance"})
axes[1].set_title("Degrés d'Appartenance Floue — FCM\n(échantillon 200 clients)")
axes[1].set_xlabel('Clients (échantillon)')
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison K-Means vs FCM
ari = adjusted_rand_score(labels_km, labels_fcm)
print("=== Comparaison K-Means vs Fuzzy C-Means ===")
print(f"\nAdjusted Rand Index (ARI) : {ari:.4f}")
print(f"  (1.0 = accord parfait, 0.0 = accord aléatoire)")
print()
print("K-Means :")
print(f"  - Clusters durs : chaque point appartient à exactement 1 cluster")
print(f"  - Rapide, déterministe")
print()
print("Fuzzy C-Means :")
print(f"  - Clusters flous : chaque point a un degré d'appartenance à chaque cluster")
print(f"  - FPC = {fpc:.4f} (plus proche de 1/3 = chevauchement important)")
print(f"  - Mieux adapté aux clients 'frontière' entre deux segments")

---
## Phase 4 : Analyse Comparative et Synthèse (10%)

In [ ]:
# === TABLEAU RÉCAPITULATIF DES PERFORMANCES ===
print("="*60)
print("SYNTHÈSE DES PERFORMANCES — APPRENTISSAGE SUPERVISÉ")
print("="*60)
print(f"\n{'Modèle':<30} {'Métrique':<20} {'Valeur':>10}")
print("-"*60)
print(f"{'Régression Linéaire':<30} {'MSE':<20} {mse:>10.4f}")
print(f"{'Régression Linéaire':<30} {'R²':<20} {r2:>10.4f}")
print(f"{'Régression Logistique':<30} {'F1-Score':<20} {f1_log:>10.4f}")
print(f"{'Régression Logistique':<30} {'Accuracy':<20} {acc_log:>10.4f}")
print(f"{'KNN (k={best_k})':<30} {'F1-Score':<20} {f1_knn:>10.4f}")
print(f"{'KNN (k={best_k})':<30} {'Accuracy':<20} {acc_knn:>10.4f}")

print()
print("="*60)
print("SYNTHÈSE — APPRENTISSAGE NON SUPERVISÉ")
print("="*60)
print(f"K-Means : k=3 clusters identifiés via Elbow Method")
print(f"FCM     : FPC={fpc:.4f}, ARI(KM vs FCM)={ari:.4f}")

In [ ]:
# === ANALYSE CRITIQUE ===
print("""
=== ANALYSE CRITIQUE DES MODÈLES ===

1. RÉGRESSION LINÉAIRE (R²=0.80)
   - Très bon score : 80% de la variance de MonthlyCharges expliquée.
   - Les features les plus influentes sont le type de contrat, l'offre internet et les services additionnels.
   - Limite : la relation entre features et prix est possiblement non-linéaire.

2. RÉGRESSION LOGISTIQUE (F1=0.59 pour Churn)
   - Accuracy globale de 79%, mais F1 modéré sur la classe minoritaire (Churn=1).
   - Cause : déséquilibre des classes (73.5% / 26.5%). 
   - Amélioration possible : class_weight='balanced' ou SMOTE pour rééquilibrer.

3. KNN (k=19, F1=0.56)
   - Performance légèrement inférieure à la régression logistique.
   - KNN est sensible à la malédiction de la dimensionnalité (19 features).
   - k=19 élevé indique que les frontières de décision sont floues.
   - Amélioration : réduction dimensionnelle (PCA) avant KNN.

4. K-MEANS (k=3)
   - 3 segments client identifiés :
     * Cluster 1 (tenure élevée, charges élevées) : clients fidèles premium
     * Cluster 2 (tenure moyenne, charges moyennes) : clients intermédiaires
     * Cluster 3 (tenure courte, faibles charges) : nouveaux clients à bas prix
   - Limite : sensible à l'initialisation, suppose des clusters sphériques.

5. FUZZY C-MEANS
   - FPC=0.35 proche de 1/c=0.33 : les clusters se chevauchent fortement.
   - Pertinent car en marketing, un client peut appartenir à plusieurs segments.
   - Plus informatif que K-Means pour cibler des actions marketing nuancées.
""")